데이터불러오기

In [1]:
import pandas as pd
import numpy as np
train_path = r"C:\Users\ghwns\Desktop\Competition\scu_ai_competition 2025\Data\campaign_train.csv"
test_path = r"C:\Users\ghwns\Desktop\Competition\scu_ai_competition 2025\Data\campaign_test.csv"
train = pd.read_csv(train_path) 
test = pd.read_csv(test_path) 

결측치 처리

In [2]:
e_level = train["고객_교육수준"].mode()[0]
m_status = train["고객_결혼여부"].mode()[0]
s_mean = train["고객_소득"].median()

e_level, m_status, s_mean

('학사', '기혼', 50827.5)

In [3]:
train["고객_교육수준"] = train["고객_교육수준"].fillna(e_level)
train["고객_결혼여부"] = train["고객_결혼여부"].fillna(m_status)
train["고객_소득"] = train["고객_소득"].fillna(s_mean)

test["고객_교육수준"] = test["고객_교육수준"].fillna(e_level)
test["고객_소득"] = test["고객_소득"].fillna(s_mean)

train.isnull().sum().sum(), test.isnull().sum().sum()

(0, 0)

피처 엔지니어링

In [ ]:
campaign_cols = ["캠페인1_수락여부", "캠페인2_수락여부", 
                 "캠페인3_수락여부", "캠페인4_수락여부", "캠페인5_수락여부"]

train["과거_캠페인_수락횟수"] = train[campaign_cols].sum(axis=1)
test["과거_캠페인_수락횟수"] = test[campaign_cols].sum(axis=1)

purchase_cols = ['고객_와인_구매금액', '고객_과일_구매금액', '고객_육류_구매금액',
                 '고객_생선_구매금액', '고객_사탕_구매금액', '고객_골드_구매금액']

train["총_구매금액"] = train[purchase_cols].sum(axis=1)
test["총_구매금액"] = test[purchase_cols].sum(axis=1)

train["와인_구매비율"] = train["고객_와인_구매금액"] / (train["총_구매금액"] + 1)
test["와인_구매비율"] = test["고객_와인_구매금액"] / (test["총_구매금액"] + 1)

train["총_웹사이트_방문"] = train["고객_매장방문_구매횟수"] + train["고객_지난달_회사사이트_방문횟수"]
test["총_웹사이트_방문"] = test["고객_매장방문_구매횟수"] + test["고객_지난달_회사사이트_방문횟수"]

ID 컬럼 및 정답 컬럼 제외

In [5]:
drop_cols = ["ID", "target", "고객_가입날짜"]

train_ft = train.drop(columns=drop_cols)
test_ft = test.drop(columns=["ID", "고객_가입날짜"])

원-핫 인코딩

In [ ]:
train_ft = train.drop(columns=["ID", "target", "고객_가입날짜"]).copy()
test_ft = test.drop(columns=["ID", "고객_가입날짜"]).copy()

from sklearn.preprocessing import OneHotEncoder
cols = train_ft.select_dtypes("object").columns.tolist()
enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

tmp = pd.DataFrame(enc.fit_transform(train_ft[cols]), columns=enc.get_feature_names_out(cols), index=train_ft.index)
train_ft = pd.concat([train_ft.drop(columns=cols), tmp], axis=1)

tmp = pd.DataFrame(enc.transform(test_ft[cols]), columns=enc.get_feature_names_out(cols), index=test_ft.index)
test_ft = pd.concat([test_ft.drop(columns=cols), tmp], axis=1)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
train_ft.loc[:, :] = scaler.fit_transform(train_ft)
test_ft.loc[:, :] = scaler.transform(test_ft)

C:\Users\ghwns\AppData\Local\Temp\ipykernel_21856\858300452.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.01305925 -0.18264026 -1.70886933 ... -0.18264026  1.1740078
  1.00442679]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_ft.loc[:, :] = scaler.fit_transform(train_ft)
C:\Users\ghwns\AppData\Local\Temp\ipykernel_21856\858300452.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 1.01987125  1.01987125  1.01987125 ...  1.01987125 -0.83494195
  1.01987125]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_ft.loc[:, :] = scaler.fit_transform(train_ft)
C:\Users\ghwns\AppData\Local\Temp\ipykernel_21856\858300452.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.

C:\Users\ghwns\AppData\Local\Temp\ipykernel_21856\858300452.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.28809301 -0.28809301 -0.28809301  3.47110122 -0.28809301 -0.28809301
 -0.28809301  3.47110122 -0.28809301 -0.28809301 -0.28809301 -0.28809301
 -0.28809301 -0.28809301 -0.28809301 -0.28809301 -0.28809301  3.47110122
 -0.28809301 -0.28809301 -0.28809301 -0.28809301 -0.28809301 -0.28809301
 -0.28809301 -0.28809301 -0.28809301 -0.28809301  3.47110122 -0.28809301
  3.47110122 -0.28809301 -0.28809301 -0.28809301 -0.28809301 -0.28809301
  3.47110122 -0.28809301 -0.28809301 -0.28809301 -0.28809301 -0.28809301
  3.47110122 -0.28809301 -0.28809301 -0.28809301 -0.28809301 -0.28809301
 -0.28809301 -0.28809301 -0.28809301 -0.28809301 -0.28809301 -0.28809301
 -0.28809301 -0.28809301 -0.28809301 -0.28809301 -0.28809301 -0.28809301
 -0.28809301 -0.28809301 -0.28809301 -0.28809301 -0.28809301 -0.28809301
 -0.28809301

스케일링

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_ft.loc[:, :] = scaler.fit_transform(train_ft)
test_ft.loc[:, :] = scaler.transform(test_ft)

모델

In [14]:
best_params = {
    'n_estimators': 500,
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': 6,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42
}

In [ ]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

SEED = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

clf_rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=SEED)
clf_gb = GradientBoostingClassifier(learning_rate=0.05, n_estimators=200, random_state=SEED)
clf_lgb = LGBMClassifier(**best_params)

# VotingClassifier (soft voting)
voting_model = VotingClassifier(
    estimators=[('rf', clf_rf), ('gb', clf_gb), ('lgb', clf_lgb)],
    voting='soft', n_jobs=-1
)

scores = cross_val_score(voting_model, train_ft, target, cv=cv, scoring='roc_auc', n_jobs=-1)
print("Voting AUC (Optuna LGB 포함):", np.mean(scores))

Voting AUC (Optuna LGB 포함): 0.8620630314870145


In [ ]:
voting_model.fit(train_ft, target)
pred = voting_model.predict_proba(test_ft)[:, 1]

submit = pd.read_csv(r"C:\Users\ghwns\Desktop\Competition\scu_ai_competition 2025\Submission\campaign_sample_submission.csv")
submit["target"] = pred
submit.to_csv(r"C:\Users\ghwns\Desktop\Competition\scu_ai_competition 2025\Submission\5.submission.csv", index=False)